# Symmetrized harmonics demo pt IV: photoionization selection rules for ePolyScat inputs
21/03/25

This notebook demonstrates functionality for working with symmetry, making use of [libmsym](https://github.com/mcodev31/libmsym) for the core symmetry functionality.

The focus here is on symmetry allowed ionization channels, and setting appropriate variables for [ePolyScat (ePS) input files](https://epolyscat.droppages.com/).

For more background, see the [Symmetriezed Haromnics demo](https://pemtk.readthedocs.io/en/latest/sym/pemtk_symHarm_demo_160322_tidy.html).

For more on point groups and symmetry, see http://www.gernot-katzers-spice-pages.com/character_tables/index.html, which includes full character tables and associated data.

## Generate basis

In [1]:
# Import class and generate basis
from pemtk.sym.symHarm import symHarm

sym = 'D10h'
symBasis = symHarm(PG = sym, lmax=4)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


* sparse not found, sparse matrix forms not available. 
* natsort not found, some sorting functions not available. 


* Setting plotter defaults with epsproc.basicPlotters.setPlotters(). Run directly to modify, or change options in local env.


* Set Holoviews with bokeh.
* pyevtk not found, VTK export not available. 
*** Mapping coeffs to ePSproc dataType = matE
Remapped dims: {'C': 'Cont', 'mu': 'it'}
Added dim Eke
Added dim Targ
Added dim Total
Added dim mu
Added dim Type
Found dipole symmetries: 
{'A2u': {'m': [0], 'pol': ['z']}, 'E1u': {'m': [-1, 1, -1, 1], 'pol': ['x', 'y']}}


/home/jovyan/github/PEMtk/pemtk/sym/_dipoleTerms.py:102: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for (col, vals) in dipolePD.iteritems():


## Set ionization channel & compute symmetries

For more options and discussion, see [the demo pt III](https://pemtk.readthedocs.io/en/latest/sym/working_with_symmetry_PEMtk_notes_180423-v210423_tidy.html#Mapping-allowed-terms-to-photoionization-matrix-elements).

In [2]:
# Specify full terms - this will indicate if a given selection fulfils the dipole selection rules for the dipole terms in the given point group.
sNeutral = 'A1g'
sIon = 'E1u'
sContinuum = 'A1g'

# The `directProductDipole` method outputs results with the dipole symmetries included
symBasis.directProductDipole([sNeutral, sIon, sContinuum])
symBasis.dipole['dipoleProducts']

{'A2u': {'m': [0],
  'pol': ['z'],
  'dipSym': 'A2u',
  'result': ['E1g'],
  'allowed': False,
  'terms': ['A1g', 'E1u', 'A1g']},
 'E1u': {'m': [-1, 1, -1, 1],
  'pol': ['x', 'y'],
  'dipSym': 'E1u',
  'result': ['A1g', 'A2g', 'E2g'],
  'allowed': True,
  'terms': ['A1g', 'E1u', 'A1g']}}

For this case we see that the set of states `[sNeutral, sIon, sContinuum]` is allowed for the $E_{1g}(x,y)$ dipole component.

In [3]:
# For cases where the continuum symmetry is not known/specified, the `directProductContinuum` method can be used, and will output results looped over all irreps
symBasis.directProductContinuum([sNeutral, sIon])

In [4]:
# For ePS, we also need to define the scattering symmetry, defined as scatSym = continuum x ion
# Run self.scatSym to compute this
symBasis.scatSym(sIon)

Adding scattering symmetries for ion=E1u


/home/jovyan/github/PEMtk/pemtk/sym/symHarm.py:350: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdInput['scat'] = scatSymList


## Output to ePS/ePSman format

Once computed, the necessary terms are output to a list of symmety tuples `(Total, Cont)`, where "Total" is the scattering symmetry, and "Cont" the corresponding continuum (photoelectron) symmetry.

In [5]:
symBasis.continuum['allowed']['ePSSymList']

[('A1u', 'E1g'),
 ('A2u', 'E1g'),
 ('E2u', 'E1g'),
 ('E1u', 'A1g'),
 ('E1u', 'A2g'),
 ('E1u', 'E2g'),
 ('E3u', 'E2g')]

Note some other lower-level methods are available, in particular `self.toePSman()` and `self.toePSmanPD()` for conversions from list or Pandas dataframes.

TODO: explicitly convert list to ePS symbols (currently this can be done in ePSman).

## Additional notes

### Multiple unpaired electrons

In some cases the initial or final state may require multiple characters, this can be handled by setting the states as lists and computing the relevant direct product(s).

In [6]:
# Setting the symmetry for the neutral and ion allows direct products to be computed, 
# and allowed terms to be determined.

sNeutral = ['B2u','B1g']
sIon = 'B2u'

In [7]:
# Check total neutral state character
symBasis.directProduct(sNeutral)

['A2u']

In [8]:
# Compute continuum results for multiple neutral UPEs - simply pass as list
symBasis.directProductContinuum([*sNeutral, sIon])

# Results are pushed to self.continuum, in dictionary and Pandas DataFrame formats, 
# and can be manipulated using standard functionality.
# The subset of allowed values are also set to a separate DataFrame and list.
symBasis.continuum['allowed']['PD']

,,allowed,m,pol,result,terms
Dipole,Target,,,,,
A2u,B2u,True,[0],[z],[A1g],"[B2u, B1g, B2u]"
E1u,E4u,True,"[-1, 1, -1, 1]","[x, y]","[A1g, A2g, E2g]","[B2u, B1g, B2u]"


## Versions

In [9]:
import scooby
scooby.Report(additional=['epsproc', 'pemtk', 'holoviews', 'hvplot', 'xarray', 'matplotlib', 'bokeh', 'pyshtools'])

--------------------------------------------------------------------------------
  Date: Fri Mar 21 12:53:26 2025 EDT

                OS : Linux
            CPU(s) : 64
           Machine : x86_64
      Architecture : 64bit
               RAM : 62.8 GiB
       Environment : Jupyter
       File system : btrfs

  Python 3.10.11 | packaged by conda-forge | (main, May 10 2023, 18:58:44)
  [GCC 11.3.0]

           epsproc : 1.3.2-dev
             pemtk : 0.0.1
         holoviews : 1.16.2
            hvplot : 0.8.4
            xarray : 2022.3.0
        matplotlib : 3.5.3
             bokeh : 3.1.1
         pyshtools : v4.10.3
             numpy : 1.23.5
             scipy : 1.10.1
           IPython : 8.13.2
            scooby : 0.7.2
--------------------------------------------------------------------------------

In [10]:
# Check current Git commit for local ePSproc version
import pemtk
from pathlib import Path
!git -C {Path(pemtk.__file__).parent} branch
!git -C {Path(pemtk.__file__).parent} log --format="%H" -n 1

* master
176ddabc2c227b4992fc6a6915875688db06ddb2


In [11]:
# Check current remote commits
!git ls-remote --heads https://github.com/phockett/pemtk

327f19a77946db80521bf593f79a49b5045d7e8d	refs/heads/master
3f4686dffdbb310f15692f978ba36d6a3d15e8d3	refs/heads/mfFittingDev
